In [25]:
# Imports
from google import genai
from google.genai import types
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd
import os
import json
import time
import base64
import re
import openpyxl
import concurrent.futures
import importlib
import prompts
importlib.reload(prompts)

<module 'prompts' from '/Users/martinhachiya/dev/nutria_backbone/nutria_solution/prompts.py'>

In [26]:
load_dotenv(override=True)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

openai_client = OpenAI()

openai_model = os.getenv("OPENAI_MODEL")

images_folder_path = os.getenv("IMG_PATH")
output_folder_path = os.getenv("OUTPUT_FOLDER_PATH")

In [27]:
def parse_json(text):
    """Parse JSON from model response with better error handling"""
    if not text:
        raise ValueError("Input text is None or empty")
    
    if not isinstance(text, str):
        raise ValueError(f"Input text is not a string, got {type(text)}")
    
    # Clean the text
    text = text.strip()
    
    # Try direct parsing first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    
    # Try to find JSON between first { and last }
    try:
        start = text.find('{')
        end = text.rfind('}')
        if start != -1 and end != -1 and end > start:
            json_str = text[start:end+1]
            return json.loads(json_str)
    except json.JSONDecodeError:
        pass
    
    # Try to extract JSON using a more robust regex
    import re
    json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
    matches = re.findall(json_pattern, text, re.DOTALL)
    
    for match in matches:
        try:
            return json.loads(match)
        except json.JSONDecodeError:
            continue
    
    # If we get here, no valid JSON was found
    raise ValueError(f"No valid JSON found in text: {text[:200]}...")
    
def count_images(folder_path):
    """Count numbered images"""
    count = 0
    index = 1
    while os.path.exists(os.path.join(folder_path, f"{index}.jpg")):
        count += 1
        index += 1
    return count

In [ ]:
def step1_gpt_serving_size(image):
    try:

        response = openai_client.chat.completions.create(
            model=openai_model,
            messages=[
            {
                "role": "system",
                "content": prompts.SYSTEM_PROMPT_PORTION_EXPERT
            },  
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompts.build_serving_estimation_prompt()},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image}"
                        }
                    }
                ]
            }],
            max_tokens=1000,
            temperature=0.1,
            response_format={"type": "json_object"}
        )

        try:
            data = parse_json(response.choices[0].message.content)
        except Exception as e:
            return {'success': False, 'error': f"No valid JSON found in the response text: {str(e)}"}
        
        return {
            'success': True,
            'description': data.get('description'),
            'serving_size': float(data.get('serving_size', 0)),
        }

    except Exception as e:
        return {'success': False, 'error': str(e)}
    
# Step 2: GPT-4 nutrition analysis
def step2_nutrition_gpt(image, description, serving_size):
    try:
        nutrition_prompt = prompts.build_nutrition_estimation_prompt(description, serving_size)

        response = openai_client.chat.completions.create(
            model=openai_model,
            messages=[
            {
                "role": "system",
                "content": prompts.SYSTEM_PROMPT_NUTRITIONIST
            },  
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": nutrition_prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{image}"
                        }
                    }
                ]
            }],
            max_tokens=1000,
            temperature=0.1,
            response_format={"type": "json_object"}
        )
        
        try:
            data = parse_json(response.choices[0].message.content)
        except Exception as e:
            return {'success': False, 'error': f"No valid JSON found in the response text: {str(e)}"}

        return {
            'success': True,
            'description': data.get('description'),
            'calories': float(data.get('calories', 0)),
            'proteins': float(data.get('proteins', 0)),
            'carbohydrates': float(data.get('carbohydrates', 0)),
            'fats': float(data.get('fats', 0)),
            'serving_size': float(data.get('serving_size', 0))
        }
    except Exception as e:
        return {'success': False, 'error': str(e)}

In [ ]:
def analyze_image_chained(image_path, index):
    """Complete chained analysis for one image"""
    start_time = time.time()
    file_name = os.path.basename(image_path)

    with open(image_path, "rb") as f:
            image_base64 = base64.b64encode(f.read()).decode("utf-8")

    print(f"\n🔄 Processing {index}: {file_name}")

    print("  Step 1: Serving size (GPT)...")
    serving_size = step1_gpt_serving_size(image_base64)

    if not serving_size['success']:
        return {'success': False, 'error': f"Step 1 failed: {serving_size['error']}", 'index': index}
    
    print(f"    → {serving_size['serving_size']}g - {serving_size['description'][:50]}...")
    if not serving_size.get("description"):
        return {'success': False, 'error': "Step 1 returned empty description", 'index': index}

    time.sleep(2)

    print("  Step 2: Nutrition analysis (GPT)...")
    nutrition = step2_nutrition_gpt(image_base64, serving_size['description'], serving_size['serving_size'])
    
    if not nutrition.get('success'):
        print(f"❌ Step 2 failed: {nutrition}")
        return {'success': False, 'error': f"Step 2 failed: {nutrition.get('error', 'Unknown error')}", 'index': index}

    print(f"    → {nutrition}")

    total_time = time.time() - start_time
    time.sleep(2)
    print(f"  ✅ Complete! {nutrition['calories']} kcal in {total_time:.1f}s")

    return {
        'success': True,
        'index': index,
        'file_name': file_name,
        'serving_size': serving_size,
        'nutrition': nutrition,
        'processing_time': total_time
    }

In [30]:
# Process dataset
def process_chained_dataset(file_name, folder_path, start=1, end=None):
    """Process images with chained analysis"""
    
    total_images = count_images(folder_path)
    if end is None:
        end = total_images
    end = min(end, total_images)
    
    print(f"🚀 Processing images {start} to {end} ({end-start+1} total)")
    
    results = []
    successful = 0
    
    for i in range(start, end + 1):
        image_path = os.path.join(folder_path, f"{i}.jpg")
        result = analyze_image_chained(image_path, i)
        results.append(result)
        
        if result['success']:
            successful += 1
    
    print(f"\n🎉 Completed! {successful}/{len(results)} successful")
    
    # Save results
    output_file = f"{file_name}.json"
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    
    print(f"📁 Results saved to: {output_file}")
    return results

In [31]:
def export_to_excel(file_name):
    """Export results to Excel"""
    with open(f"{file_name}.json", 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Prepare final results
    final_results = []
    for item in data:
        if item['success']:
            nutrition = item['nutrition']
            final_results.append({
                'id': item['index'],
                'description': nutrition['description'],
                'serving_size': nutrition['serving_size'],
                'calories': nutrition['calories'],
                'proteins': nutrition['proteins'],
                'carbohydrates': nutrition['carbohydrates'],
                'fats': nutrition['fats']
            })
    
    # Create DataFrame and export
    df = pd.DataFrame(final_results)
    output_path = os.path.join(output_folder_path, f"{file_name}.xlsx")
    df.to_excel(output_path, index=False)
    
    print(f"✅ Excel exported: {output_path}")
    print(f"📊 {len(final_results)} successful analyses")
    
    return output_path

In [32]:
file_name = "nutria_gpt"

In [33]:
results = process_chained_dataset(file_name, images_folder_path, start=1, end=count_images(images_folder_path))
# results = process_chained_dataset(file_name, images_folder_path, start=1, end=2)

🚀 Processing images 1 to 50 (50 total)

🔄 Processing 1: 1.jpg
  Step 1: Serving size (Gemini)...
    → 300.0g - Vaso de cerveza rubia...
  Step 2: Nutrition analysis (Gemini)...
    → {'success': True, 'description': 'Vaso de cerveza rubia', 'calories': 129.0, 'proteins': 1.5, 'carbohydrates': 10.5, 'fats': 0.0, 'serving_size': 300.0}
  ✅ Complete! 129.0 kcal in 6.8s

🔄 Processing 2: 2.jpg
  Step 1: Serving size (Gemini)...
    → 300.0g - Pasta tipo penne con salsa de tomate y albahaca...
  Step 2: Nutrition analysis (Gemini)...
    → {'success': True, 'description': 'Pasta tipo penne con salsa de tomate y albahaca', 'calories': 390.0, 'proteins': 13.5, 'carbohydrates': 75.0, 'fats': 3.5, 'serving_size': 300.0}
  ✅ Complete! 390.0 kcal in 7.1s

🔄 Processing 3: 3.jpg
  Step 1: Serving size (Gemini)...
    → 180.0g - Croissant con semillas, huevo revuelto, salmón ahu...
  Step 2: Nutrition analysis (Gemini)...
    → {'success': True, 'description': 'Croissant con semillas, huevo revuelto

In [34]:
print(results)

[{'success': True, 'index': 1, 'file_name': '1.jpg', 'serving_size': {'success': True, 'description': 'Vaso de cerveza rubia', 'serving_size': 300.0}, 'nutrition': {'success': True, 'description': 'Vaso de cerveza rubia', 'calories': 129.0, 'proteins': 1.5, 'carbohydrates': 10.5, 'fats': 0.0, 'serving_size': 300.0}, 'processing_time': 6.805786848068237}, {'success': True, 'index': 2, 'file_name': '2.jpg', 'serving_size': {'success': True, 'description': 'Pasta tipo penne con salsa de tomate y albahaca', 'serving_size': 300.0}, 'nutrition': {'success': True, 'description': 'Pasta tipo penne con salsa de tomate y albahaca', 'calories': 390.0, 'proteins': 13.5, 'carbohydrates': 75.0, 'fats': 3.5, 'serving_size': 300.0}, 'processing_time': 7.09154200553894}, {'success': True, 'index': 3, 'file_name': '3.jpg', 'serving_size': {'success': True, 'description': 'Croissant con semillas, huevo revuelto, salmón ahumado y tomates cherry', 'serving_size': 180.0}, 'nutrition': {'success': True, 'des

In [35]:
excel_path = export_to_excel(file_name)
print(f"\n🎯 Done! Check: {excel_path}")

✅ Excel exported: /Users/martinhachiya/dev/datasets/nutria_backbone/llm_response/nutria_gpt.xlsx
📊 50 successful analyses

🎯 Done! Check: /Users/martinhachiya/dev/datasets/nutria_backbone/llm_response/nutria_gpt.xlsx
